# ELEC6604 Assignment 3 — CNN_A (Google Colab)

Run the cells in order. In Colab select **Runtime → Change runtime type → GPU** before training.
This notebook currently implements **CNN_A only**: a plain CNN with 356,810 parameters and no regularization.
CNN_B and CNN_C will be added later. No experiment results are pre-filled.

The default experiment runs three seeds for 30 epochs each. For a quick check set `SEEDS = [42]` and `EPOCHS = 1`, then restore the defaults for reported results.
The official test set is used only for final evaluation; validation data comes from the original training set.


In [ ]:
import os
import json
import time
import random
import shutil
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.model_selection import train_test_split

print("TensorFlow:", tf.__version__)
print("GPUs:", tf.config.list_physical_devices("GPU"))
if not tf.config.list_physical_devices("GPU"):
    print("No GPU detected. Enable a GPU runtime in Colab for faster training.")


## 1. Experiment settings
Keep the validation split fixed across seeds, so repeats measure training variability.
`NORMALIZATION` can be `zero_one` or `minus_one_one`. To compare normalization later, rerun with the other value while keeping all other settings fixed.
Enable Drive saving to retain outputs after the Colab runtime disconnects. Otherwise download the results ZIP before ending the session.


In [ ]:
SEEDS = [42, 123, 2026]
SPLIT_SEED = 2026
EPOCHS = 30
BATCH_SIZE = 128
LEARNING_RATE = 1e-3
NORMALIZATION = "zero_one"
SAVE_TO_DRIVE = False

if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    OUTPUT_ROOT = Path("/content/drive/MyDrive/ELEC6604_Assignment3")
else:
    OUTPUT_ROOT = Path("/content/assignment3_results")

RUN_ID = time.strftime("%Y%m%d-%H%M%S")
OUTPUT_DIR = OUTPUT_ROOT / f"CNN_A_{NORMALIZATION}_{RUN_ID}"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
assert NORMALIZATION in {"zero_one", "minus_one_one"}
assert EPOCHS > 0 and len(SEEDS) == len(set(SEEDS)) and len(SEEDS) > 0
try:
    tf.config.experimental.enable_op_determinism()
except (AttributeError, RuntimeError) as exc:
    print("Deterministic operations could not be enabled:", exc)
print("Results directory:", OUTPUT_DIR)


## 2. Load and preprocess CIFAR-10
Download directly using Keras; no manually uploaded CIFAR batch files are needed.
Split 50,000 original training images into 45,000 training and 5,000 validation images, stratified by class.
Pixel values are converted to float32 and normalized. Labels are one-hot encoded for categorical cross-entropy.


In [ ]:
(images, labels), (test_images, test_labels) = keras.datasets.cifar10.load_data()
train_indices, val_indices = train_test_split(
    np.arange(len(images)), test_size=0.1, random_state=SPLIT_SEED,
    stratify=labels.ravel(),
)

def normalize(x):
    x = x.astype("float32") / 255.0
    return (x - 0.5) * 2.0 if NORMALIZATION == "minus_one_one" else x

X_train = normalize(images[train_indices])
X_val = normalize(images[val_indices])
X_test = normalize(test_images)
Y_train = keras.utils.to_categorical(labels[train_indices], 10)
Y_val = keras.utils.to_categorical(labels[val_indices], 10)
Y_test = keras.utils.to_categorical(test_labels, 10)
np.savez(OUTPUT_DIR / "split_indices.npz", train=train_indices, validation=val_indices)
del images, test_images
for name, x, y in [("Train", X_train, Y_train), ("Validation", X_val, Y_val), ("Test", X_test, Y_test)]:
    print(name, "images:", x.shape, "labels:", y.shape, "range:", (x.min(), x.max()))
assert (len(X_train), len(X_val), len(X_test)) == (45000, 5000, 10000)
assert not np.intersect1d(train_indices, val_indices).size


## 3. CNN_A architecture
`32×32×3 → Conv32 → Pool → Conv64 → Pool → Conv128 → Pool → Flatten → Dense128 → Softmax10`

All convolutions use 3×3 kernels, same padding and ReLU; pooling uses 2×2 windows.
There is **no Dropout, Batch Normalization, weight decay, data augmentation or label smoothing**.
Parameter calculation: 896 + 18,496 + 73,856 + 262,272 + 1,290 = **356,810**, within the required 200,000–700,000 range.


In [ ]:
def build_cnn_a():
    model = keras.Sequential([
        layers.Input(shape=(32, 32, 3)),
        layers.Conv2D(32, 3, padding="same", activation="relu", name="conv1"),
        layers.MaxPooling2D(2, name="pool1"),
        layers.Conv2D(64, 3, padding="same", activation="relu", name="conv2"),
        layers.MaxPooling2D(2, name="pool2"),
        layers.Conv2D(128, 3, padding="same", activation="relu", name="conv3"),
        layers.MaxPooling2D(2, name="pool3"),
        layers.Flatten(name="flatten"),
        layers.Dense(128, activation="relu", name="dense1"),
        layers.Dense(10, activation="softmax", name="class_probabilities"),
    ], name="CNN_A")
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=LEARNING_RATE),
        loss="categorical_crossentropy", metrics=["accuracy"],
    )
    return model

preview_model = build_cnn_a()
preview_model.summary()
assert preview_model.count_params() == 356810
assert 200_000 <= preview_model.count_params() <= 700_000
summary_lines = []
preview_model.summary(print_fn=lambda line: summary_lines.append(line))
(OUTPUT_DIR / "model_summary.txt").write_text("\n".join(summary_lines))
try:
    keras.utils.plot_model(preview_model, to_file=str(OUTPUT_DIR / "CNN_A_architecture.png"), show_shapes=True)
except (ImportError, OSError) as exc:
    print("Optional architecture image unavailable:", exc)
del preview_model


## 4. Train and evaluate three seeds
Each run uses a fresh model and optimizer. All runs complete the same number of epochs (no early stopping).
The checkpoint with the lowest validation loss is selected for final evaluation, without consulting test results.
Training time includes validation and checkpoint writing, but excludes data download and final evaluation.
Per-epoch training metrics are measured during updates; the final train/validation metrics below evaluate the selected checkpoint in inference mode.


In [ ]:
config = {
    "model": "CNN_A", "seeds": SEEDS, "split_seed": SPLIT_SEED,
    "epochs": EPOCHS, "batch_size": BATCH_SIZE, "learning_rate": LEARNING_RATE,
    "normalization": NORMALIZATION, "tensorflow_version": tf.__version__,
    "devices": [str(d) for d in tf.config.list_physical_devices()],
    "checkpoint_selection": "minimum validation loss",
}
(OUTPUT_DIR / "config.json").write_text(json.dumps(config, indent=2))
results, histories = [], {}
for seed in SEEDS:
    keras.backend.clear_session()
    keras.utils.set_random_seed(seed)
    model = build_cnn_a()
    run_dir = OUTPUT_DIR / f"seed_{seed}"
    run_dir.mkdir(exist_ok=True)
    checkpoint = run_dir / "best.weights.h5"
    print(f"\nTraining CNN_A: seed={seed}")
    start = time.perf_counter()
    history = model.fit(
        X_train, Y_train, validation_data=(X_val, Y_val),
        epochs=EPOCHS, batch_size=BATCH_SIZE, shuffle=True, verbose=2,
        callbacks=[
            keras.callbacks.ModelCheckpoint(str(checkpoint), monitor="val_loss", save_best_only=True, save_weights_only=True),
            keras.callbacks.CSVLogger(str(run_dir / "epochs.csv")),
            keras.callbacks.TerminateOnNaN(),
        ],
    )
    training_seconds = time.perf_counter() - start
    if len(history.history["loss"]) != EPOCHS or not all(
        np.isfinite(values).all() for values in history.history.values()
    ):
        raise RuntimeError("Training failed or produced non-finite metrics; inspect the run before reporting results.")
    histories[seed] = history.history
    (run_dir / "history.json").write_text(json.dumps(history.history, indent=2))
    model.load_weights(checkpoint)
    train_metrics = model.evaluate(X_train, Y_train, batch_size=BATCH_SIZE, verbose=0, return_dict=True)
    val_metrics = model.evaluate(X_val, Y_val, batch_size=BATCH_SIZE, verbose=0, return_dict=True)
    test_metrics = model.evaluate(X_test, Y_test, batch_size=BATCH_SIZE, verbose=0, return_dict=True)
    row = {
        "seed": seed, "parameters": model.count_params(),
        "best_epoch": int(np.argmin(history.history["val_loss"]) + 1),
        "epochs_run": len(history.history["loss"]), "training_seconds": training_seconds,
        "seconds_per_epoch": training_seconds / len(history.history["loss"]),
        "train_accuracy": train_metrics["accuracy"], "train_loss": train_metrics["loss"],
        "val_accuracy": val_metrics["accuracy"], "val_loss": val_metrics["loss"],
        "test_accuracy": test_metrics["accuracy"], "test_loss": test_metrics["loss"],
        "train_val_accuracy_gap": train_metrics["accuracy"] - val_metrics["accuracy"],
    }
    results.append(row)
    pd.DataFrame(results).to_csv(OUTPUT_DIR / "results_by_seed.csv", index=False)
    print(row)


## 5. Summarize and plot
Accuracy is stored as a fraction (multiply by 100 for percent). Standard deviation uses the sample definition (`ddof=1`).
With only one smoke-test seed the standard deviation is undefined; these are not the final three-seed results.
Use the curves and train–validation gap to discuss overfitting; do not assume it occurred before examining the results.


In [ ]:
results_df = pd.DataFrame(results)
display(results_df)
metric_columns = ["train_accuracy", "train_loss", "val_accuracy", "val_loss", "test_accuracy", "test_loss", "training_seconds", "seconds_per_epoch", "train_val_accuracy_gap"]
summary_df = results_df[metric_columns].agg(["mean", "std"]).T
summary_df.to_csv(OUTPUT_DIR / "summary_mean_std.csv")
display(summary_df)
if len(SEEDS) != 3:
    print("This is not the required three-seed final experiment.")

fig, axes = plt.subplots(len(SEEDS), 2, figsize=(12, 4 * len(SEEDS)), squeeze=False)
for i, seed in enumerate(SEEDS):
    h = histories[seed]
    epochs = np.arange(1, len(h["loss"]) + 1)
    for j, metric in enumerate(["accuracy", "loss"]):
        ax = axes[i, j]
        ax.plot(epochs, h[metric], label="Training")
        ax.plot(epochs, h["val_" + metric], label="Validation")
        ax.axvline(results[i]["best_epoch"], color="gray", linestyle="--", label="Selected epoch")
        ax.set(title=f"CNN_A — seed {seed}", xlabel="Epoch", ylabel=metric.capitalize())
        if metric == "accuracy":
            ax.set_ylim(0, 1)
        ax.legend()
        ax.grid(alpha=0.2)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "CNN_A_learning_curves.png", dpi=180, bbox_inches="tight")
plt.show()

archive = shutil.make_archive(str(OUTPUT_DIR), "zip", root_dir=OUTPUT_DIR)
print("Saved experiment ZIP:", archive)


In [ ]:
# Run this cell to download the ZIP to your computer.
from google.colab import files
files.download(archive)


## Report checklist — CNN_A
- State architecture, parameter count, optimizer, learning rate, batch size, normalization and epochs.
- Include the architecture diagram and training/validation curves on the CNN_A report page.
- Report final test accuracy and three-seed mean ± standard deviation, with training time and hardware.
- Explain the checkpoint selection rule and any observed overfitting using actual results.
- CNN_B, CNN_C, the normalization comparison and the complete six-page report remain separate work.
